In [1]:
import cv2
import threading
import numpy as np
import playsound
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

def play_alarm_sound():
        playsound.playsound("alarmsound.mp3")
alarm_status = False
fire_reported = 0
video = cv2.VideoCapture("firevid.mp4")
def send_email_alert():
    sender_email = "tesstt.127@gmail.com"
    sender_password = "cjzy ozeq xmjb viml"
    receiver_email = "areeba0246@gmail.com"

    subject = "🔥 Fire Alert Detected!"
    body = "Fire has been detected by the security camera system. Please take immediate action."

    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = subject

    msg.attach(MIMEText(body, 'plain'))

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        server.quit()
        print("📧 Email sent!")
    except Exception as e:
        print("❌ Failed to send email:", str(e))
while True:
    grabbed, frame = video.read()
    if not grabbed:
        break
    frame = cv2.resize(frame, (960,540))
    blur = cv2.GaussianBlur(frame, (21,21),0)
    hsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)
    lower_red = np.array([10,150,50])
    upper_red = np.array([35,255,255])
    mask = cv2.inRange(hsv, lower_red, upper_red)
    output = cv2.bitwise_and(frame , frame , mask=mask)
    no_red = cv2.countNonZero(mask)
    if int(no_red) > 15000:
        fire_reported +=1
    cv2.imshow("fire detected" ,output)
    if fire_reported >= 1 and not alarm_status:
        threading.Thread(target = play_alarm_sound).start()
        threading.Thread(target=send_email_alert).start()
        alarm_status = True
        
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cv2.destroyAllWindows()
video.release()

📧 Email sent!
